In [ ]:
# =============================================================
# CELL 1: Install Dependencies
# =============================================================

!pip install google-adk
!pip install google-cloud-aiplatform[agent_engines,adk]
!pip install requests python-dotenv

In [ ]:
# =============================================================
# CELL 2: Configuration
# =============================================================

PROJECT_ID = "qwiklabs-gcp-01-171e30612222"
LOCATION = "us-central1"  # This is what worked in your MVP

In [ ]:
# =============================================================
# CELL 3: Setup
# =============================================================

import subprocess
import os
import time

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging-{LOCATION}"

print("1. Enabling APIs...")
subprocess.run(["gcloud", "services", "enable", "aiplatform.googleapis.com",
    f"--project={PROJECT_ID}"], capture_output=True, text=True)
print("   Done.")

print(f"2. Staging bucket: {STAGING_BUCKET}")
result = subprocess.run(["gsutil", "mb", "-l", LOCATION, STAGING_BUCKET],
    capture_output=True, text=True)
print(f"   {'Ready' if 'already' in result.stderr.lower() or result.returncode == 0 else result.stderr.strip()}")

print("3. Project number...")
result = subprocess.run(["gcloud", "projects", "describe", PROJECT_ID,
    "--format=value(projectNumber)"], capture_output=True, text=True)
PROJECT_NUMBER = result.stdout.strip()
print(f"   {PROJECT_NUMBER}")

print("4. Ensuring Vertex AI service agent exists...")
subprocess.run(["gcloud", "beta", "services", "identity", "create",
    "--service=aiplatform.googleapis.com", f"--project={PROJECT_ID}"],
    capture_output=True, text=True)

SA = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
print(f"   Granting roles/aiplatform.user to {SA}...")
subprocess.run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
    f"--member=serviceAccount:{SA}", "--role=roles/aiplatform.user", "--quiet"],
    capture_output=True, text=True)
print(f"   Granting roles/run.admin to {SA}...")
subprocess.run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
    f"--member=serviceAccount:{SA}", "--role=roles/run.admin", "--quiet"],
    capture_output=True, text=True)

print(f"\nSetup complete. Please wait ~10 seconds before deploying.")

1. Enabling APIs...
   Done.
2. Staging bucket: gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1
   Ready
3. Project number...
   922001826607
4. Ensuring Vertex AI service agent exists...
   Granting roles/aiplatform.user to service-922001826607@gcp-sa-aiplatform-re.iam.gserviceaccount.com...
   Granting roles/run.admin to service-922001826607@gcp-sa-aiplatform-re.iam.gserviceaccount.com...

Setup complete. Please wait ~10 seconds before deploying.


In [ ]:
# =============================================================
# CELL 4: Initialize Vertex AI
# =============================================================

import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Vertex AI initialized: {PROJECT_ID} / {LOCATION}")

Vertex AI initialized: qwiklabs-gcp-01-171e30612222 / us-central1


In [ ]:
# =============================================================
# CELL 5: Clean up existing deployments
# =============================================================

from vertexai import agent_engines
import time

print("Checking for existing deployments to clean up...")
try:
    count = 0
    for existing in agent_engines.list():
        rn = getattr(existing, 'resource_name', None) or getattr(existing, 'name', None)
        dn = getattr(existing, 'display_name', 'unnamed')
        print(f"  Found: {dn} -> {rn}")

        for attempt in range(3):
            try:
                agent_engines.delete(rn, force=True)
                print(f"  Deleted.")
                count += 1
                break
            except TypeError:
                try:
                    agent_engines.delete(rn)
                    print(f"  Deleted.")
                    count += 1
                    break
                except Exception as e2:
                    print(f"  Attempt {attempt+1}: {e2}")
                    time.sleep(5)
            except Exception as e:
                print(f"  Attempt {attempt+1}: {e}")
                time.sleep(5)

    if count == 0:
        print("  No existing deployments found. Clean slate.")
    else:
        print(f"\n  Deleted {count} deployment(s). Waiting 30 seconds...")
        time.sleep(30)
        print("  Ready to deploy.")
except Exception as e:
    print(f"  List failed: {e}")
    print("  Proceeding anyway — may be a clean slate.")

INFO:vertexai.agent_engines:Deleting AgentEngine resource: projects/922001826607/locations/us-central1/reasoningEngines/4776092693593849856


Checking for existing deployments to clean up...
  Found:  -> projects/922001826607/locations/us-central1/reasoningEngines/4776092693593849856


INFO:vertexai.agent_engines:Delete AgentEngine backing LRO: projects/922001826607/locations/us-central1/operations/1318512774703218688
INFO:vertexai.agent_engines:AgentEngine resource deleted: projects/922001826607/locations/us-central1/reasoningEngines/4776092693593849856
INFO:vertexai.agent_engines:Deleting AgentEngine resource: projects/922001826607/locations/us-central1/reasoningEngines/7097698301503340544


  Deleted.
  Found:  -> projects/922001826607/locations/us-central1/reasoningEngines/7097698301503340544


INFO:vertexai.agent_engines:Delete AgentEngine backing LRO: projects/922001826607/locations/us-central1/operations/4142269741064519680
INFO:vertexai.agent_engines:AgentEngine resource deleted: projects/922001826607/locations/us-central1/reasoningEngines/7097698301503340544


  Deleted.

  Deleted 2 deployment(s). Waiting 30 seconds...
  Ready to deploy.


In [ ]:
# =============================================================
# CELL 6: Define, Test, and Deploy GeoGuesser (FULL FEATURED)
# =============================================================

import time

def deploy_geoguesser():
    """
    Creates the GeoGuesser agent with geographic knowledge tools
    and deploys to Agent Engine.
    """
    import requests
    from google.adk.agents import LlmAgent
    from vertexai.preview.reasoning_engines import AdkApp
    from vertexai import agent_engines

    # --- TOOL 1: Wikidata (FIXED — added User-Agent header) ---
    def search_wikidata(query: str) -> str:
        """Searches Wikidata for structured geographic information about a place.

        Args:
            query: A search term describing a place, landmark, or geographic feature.

        Returns:
            A string containing structured data about matching places.
        """
        try:
            url = "https://www.wikidata.org/w/api.php"
            params = {
                "action": "wbsearchentities",
                "search": query,
                "language": "en",
                "format": "json",
                "limit": 5,
                "type": "item"
            }
            headers = {
                "User-Agent": "GeoGuesserBot/1.0 (educational project; contact: student@example.com)"
            }
            resp = requests.get(url, params=params, headers=headers, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            results = []
            for item in data.get("search", []):
                label = item.get("label", "Unknown")
                desc = item.get("description", "No description")
                wikidata_id = item.get("id", "")
                results.append(f"- {label} ({wikidata_id}): {desc}")
            if results:
                return f"Wikidata results for '{query}':\n" + "\n".join(results)
            return f"No Wikidata results for '{query}'."
        except Exception as e:
            return f"Wikidata error: {e}"

    # --- TOOL 2: OpenStreetMap Nominatim ---

    def search_places(query: str) -> str:
        """Searches for places and geographic locations using multiple sources.

        Args:
            query: A place name or geographic description to search for.

        Returns:
            A string containing matching places with details.
        """
        try:
            headers = {
                "User-Agent": "GeoGuesserBot/1.0 (educational project; contact: student@example.com)",
                "Accept": "application/sparql-results+json"
            }

            # Use Wikidata SPARQL for geographic place search
            # This is more reliable from cloud IPs than Nominatim
            sparql_query = f"""
            SELECT ?place ?placeLabel ?countryLabel ?coord WHERE {{
              ?place wdt:P31/wdt:P279* wd:Q486972.
              ?place rdfs:label ?placeLabel.
              FILTER(LANG(?placeLabel) = "en").
              FILTER(CONTAINS(LCASE(?placeLabel), LCASE("{query}"))).
              OPTIONAL {{ ?place wdt:P17 ?country. }}
              OPTIONAL {{ ?place wdt:P625 ?coord. }}
              SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
            }}
            LIMIT 5
            """

            sparql_url = "https://query.wikidata.org/sparql"
            resp = requests.get(
                sparql_url,
                params={"query": sparql_query, "format": "json"},
                headers=headers,
                timeout=15
            )
            resp.raise_for_status()
            data = resp.json()
            bindings = data.get("results", {}).get("bindings", [])

            if bindings:
                results = []
                for b in bindings:
                    place = b.get("placeLabel", {}).get("value", "Unknown")
                    country = b.get("countryLabel", {}).get("value", "")
                    coord = b.get("coord", {}).get("value", "")
                    line = f"- {place}"
                    if country:
                        line += f", {country}"
                    if coord:
                        line += f" (coords: {coord})"
                    results.append(line)
                return f"Place search results for '{query}':\n" + "\n".join(results)

            # Fallback: use Wikidata text search
            fallback_url = "https://www.wikidata.org/w/api.php"
            fallback_params = {
                "action": "wbsearchentities",
                "search": query,
                "language": "en",
                "format": "json",
                "limit": 5,
                "type": "item"
            }
            resp2 = requests.get(
                fallback_url,
                params=fallback_params,
                headers=headers,
                timeout=10
            )
            resp2.raise_for_status()
            data2 = resp2.json()
            results2 = []
            for item in data2.get("search", []):
                label = item.get("label", "Unknown")
                desc = item.get("description", "")
                if any(word in desc.lower() for word in
                       ["city", "town", "village", "country", "region",
                        "island", "mountain", "river", "place", "municipality"]):
                    results2.append(f"- {label}: {desc}")
            if results2:
                return f"Place search results for '{query}':\n" + "\n".join(results2)

            return f"No places found for '{query}'."

        except Exception as e:
            return f"Place search error: {e}"

    # --- TOOL 3: REST Countries ---
    def lookup_country(country_name: str) -> str:
        """Looks up detailed information about a country.

        Args:
            country_name: The name of a country to look up.

        Returns:
            A string with country details: capital, region, languages, population.
        """
        try:
            url = f"https://restcountries.com/v3.1/name/{country_name}"
            headers = {
                "User-Agent": "GeoGuesserBot/1.0 (educational project; contact: student@example.com)"
            }
            resp = requests.get(url, headers=headers, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            if not data:
                return f"No data for '{country_name}'."
            c = data[0]
            name = c.get("name", {}).get("common", "Unknown")
            capital = ", ".join(c.get("capital", ["Unknown"]))
            region = c.get("region", "Unknown")
            subregion = c.get("subregion", "Unknown")
            pop = c.get("population", 0)
            langs = ", ".join(c.get("languages", {}).values()) if c.get("languages") else "Unknown"
            currencies = ", ".join(
                f"{v.get('name', '?')} ({v.get('symbol', '?')})"
                for v in c.get("currencies", {}).values()
            ) if c.get("currencies") else "Unknown"
            return (f"Country: {name}\nCapital: {capital}\n"
                    f"Region: {region}/{subregion}\n"
                    f"Population: {pop:,}\nLanguages: {langs}\n"
                    f"Currencies: {currencies}")
        except Exception as e:
            return f"Country lookup error: {e}"

    # --- DEFINE THE AGENT ---
    agent = LlmAgent(
        name=f"geoguesser_{int(time.time())}",
        model="gemini-2.0-flash",
        instruction="""You are GeoGuesser Bot 🌍 — an interactive location guessing game!

You have THREE geographic research tools:
1. search_wikidata — structured facts from Wikidata about places, landmarks, and features
2. search_places — OpenStreetMap place search with coordinates and types
3. lookup_country — country details (languages, capital, region, population, currencies)

═══════════════════════════════════════
GAME RULES
═══════════════════════════════════════

The user describes a place. You try to guess it. The game continues
until you guess correctly OR the user says "quit", "give up", or "new game".

There is NO LIMIT to how many guesses you can make. Keep trying!

═══════════════════════════════════════
FOR EACH GUESS, FOLLOW THIS PROCESS:
═══════════════════════════════════════

You get up to 10 RESEARCH ROUNDS per guess attempt. Use them wisely.

ROUND 1: Extract ALL clues from the user's description. Search using
multiple combinations of clues with your tools. Identify 3-5 candidates.

ROUND 2-3: Cross-reference candidates. Use search_wikidata to verify
landmarks. Use search_places to confirm locations exist. Use
lookup_country to check language/culture clues.

ROUND 4-10 (as needed): Drill into specific clues that haven't been
resolved. Search for the most distinctive or unusual clues specifically.
Compare remaining candidates against unmatched clues. Stop researching
once you reach HIGH confidence.

AFTER RESEARCH, present EXACTLY ONE guess:

═══════════════════════════════════════
🌍 MY GUESS: [Location, Country/Region]
═══════════════════════════════════════
CONFIDENCE: [✅ HIGH / 🟡 MEDIUM / 🔴 LOW]

CLUE SCORECARD:
✅ [clue] — [how it matches]
✅ [clue] — [how it matches]
❌ [clue] — [doesn't match] (if any)

SCORE: [X/Y clues matched]
RESEARCH ROUNDS USED: [N/10]

FUN FACT: [something interesting about this place]
═══════════════════════════════════════
Did I get it right? 🎯

═══════════════════════════════════════
HANDLING USER RESPONSES
═══════════════════════════════════════

IF USER SAYS "yes", "correct", "right", "you got it":
→ Celebrate! "🎉 Got it! That was [easy/tricky/challenging]!"
→ Ask: "Want to play again? Describe another place!"

IF USER SAYS "no", "wrong", "nope", "not quite":
→ "Not yet! Can you give me another clue? What about the climate,
   food, architecture, or what language people speak there?"
→ When they provide more clues, research again with ALL clues
   (original description + all new clues combined)
→ Make a NEW guess using the expanded clue set

IF USER SAYS "quit", "give up", "stop", "exit":
→ "Thanks for playing! 🌍 What was the answer? I'd love to know!"

IF USER SAYS "new game", "new", "reset", "another":
→ "🔄 New game! Describe a place and I'll start guessing!"

IF USER SAYS "hint" or "clue":
→ "I need YOU to give ME clues! Describe things like the landscape,
   weather, food, buildings, language, or famous landmarks."

═══════════════════════════════════════
GREETING
═══════════════════════════════════════

When the user first says hello or asks how to play:
"Welcome to GeoGuesser Bot! 🌍

I'll try to guess any place in the world based on your description!

HOW TO PLAY:
🗣️ Describe a place (landmarks, food, weather, culture, etc.)
🔍 I'll research using Wikidata, OpenStreetMap, and country databases
🌍 I'll make my best guess
❌ Wrong? Give me more clues and I'll try again!
✅ Right? Celebrate and play again!

I can do up to 10 rounds of research per guess and I never give up!
Type 'quit' to end or 'new game' to start fresh.

Ready? Describe your place!"

Be enthusiastic, educational, and fun! This is a game!""",
        tools=[search_wikidata, search_places, lookup_country],
    )
    print(f"Agent created: {agent.name}")

    # --- WRAP AND TEST LOCALLY ---
    app = AdkApp(agent=agent)
    print("AdkApp created.\n")

    print("LOCAL TEST:")
    print("-" * 40)
    try:
        for event in app.stream_query(
            user_id="local-test",
            message="A place with a massive iron tower, famous cafes, a river, romantic language, incredible pastries.",
        ):
            print(event)
        print("-" * 40)
        print("Local test complete.\n")
    except Exception as e:
        print(f"Local test error: {e}")
        print("Proceeding to deploy anyway.\n")

    # --- DEPLOY ---
    print("Deploying to Agent Engine...")
    print("This may take 5-10 minutes...\n")

    remote = agent_engines.create(
        app,
        requirements=[
            "google-cloud-aiplatform[agent_engines,adk]",
            "requests",
        ],
    )

    print("=" * 60)
    print("DEPLOYMENT SUCCESSFUL!")
    print("=" * 60)
    print(f"Resource: {remote.resource_name}")
    return remote


remote_agent = deploy_geoguesser()

Agent created: geoguesser_1772833619
AdkApp created.

LOCAL TEST:
----------------------------------------


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requ

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Welcome to GeoGuesser Bot! 🌍\n\nI'll try to guess any place in the world based on your description!\n\nHOW TO PLAY:\n🗣️ Describe a place (landmarks, food, weather, culture, etc.)\n🔍 I'll research using Wikidata, OpenStreetMap, and country databases\n🌍 I'll make my best guess\n❌ Wrong? Give me more clues and I'll try again!\n✅ Right? Celebrate and play again!\n\nI can do up to 10 rounds of research per guess and I never give up!\nType 'quit' to end or 'new game' to start fresh.\n\nReady? Describe your place!\n"}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 140, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 140}], 'prompt_token_count': 1202, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 1202}], 'total_token_count': 1342, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': 6.650119238266988e-07, 'invocation_id': 'e-76cf48dc-f985-400a-bc97-ee429dc

INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-01-171e30612222-agent-staging-us-central1/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/922001826607/locations/us-central1/reasoningEngines/6822978724233740288/operations/6089513629948837888
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-01-171e30612222
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/922001826607/locations/us-central1/reasoningEngines/6822978724233740288
INFO:vertexai.agent_engines:To use thi

DEPLOYMENT SUCCESSFUL!
Resource: projects/922001826607/locations/us-central1/reasoningEngines/6822978724233740288


In [ ]:
# =============================================================
# CELL 7: Test — Easy Location (Paris)
# =============================================================

print("Test 1: Easy location (Paris)")
print("=" * 60)
print()

for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="A place with a massive iron tower built for a world's fair, famous sidewalk cafes, a river running through it, a romantic language, and incredible pastries.",
):
    print(event)

Test 1: Easy location (Paris)

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Okay, I think I have a good idea where this is!\n\nROUND 1/10\nLet's start by gathering some information about the clues provided.\n\n"}, {'function_call': {'id': 'adk-6fce36b2-d032-4f0a-97f9-134a2dadbe8d', 'args': {'query': "iron tower world's fair"}, 'name': 'search_wikidata'}}, {'function_call': {'id': 'adk-f8a5fd6c-a801-4921-97ce-cc2f5491652b', 'args': {'query': 'sidewalk cafes river romantic language pastries'}, 'name': 'search_places'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 56, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 56}], 'prompt_token_count': 1215, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 1215}], 'total_token_count': 1271, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.19534170627593994, 'invocation_id': 'e-05a9e6d6-7491-4781-abcd-6fb3e263a4e4', 'author': 'geoguesser_1772833619',

In [ ]:
# =============================================================
# CELL 8: Test — Medium Location (Uluru)
# =============================================================

print("Test 2: Medium location (Uluru)")
print("=" * 60)
print()

for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Red desert, ancient sacred rock formations, extremely remote, incredible stars, hot days cold nights, indigenous culture tens of thousands of years old.",
):
    print(event)

Test 2: Medium location (Uluru)

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Welcome to GeoGuesser Bot! 🌍\n\nI'll try to guess any place in the world based on your description!\n\nHOW TO PLAY:\n🗣️ Describe a place (landmarks, food, weather, culture, etc.)\n🔍 I'll research using Wikidata, OpenStreetMap, and country databases\n🌍 I'll make my best guess\n❌ Wrong? Give me more clues and I'll try again!\n✅ Right? Celebrate and play again!\n\nI can do up to 10 rounds of research per guess and I never give up!\nType 'quit' to end or 'new game' to start fresh.\n\nReady? Describe your place!\n"}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 140, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 140}], 'prompt_token_count': 1210, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 1210}], 'total_token_count': 1350, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -3.11158925926845e-07, 'invocation_id': '

In [ ]:
# =============================================================
# CELL 9: Test — Hard Location (Lisbon)
# =============================================================

print("Test 3: Hard location (Lisbon)")
print("=" * 60)
print()

for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Coastal city with colorful houses on hills, famous custard tarts, trams in narrow streets, street art everywhere, a large red bridge that is NOT the Golden Gate.",
):
    print(event)

Test 3: Hard location (Lisbon)

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Okay, I'm on it! This sounds like a fun challenge.\n\nROUND 1: Initial Research\n\nI'll start by searching for some of the key clues: coastal city, colorful houses, custard tarts, trams, street art, and a red bridge.\n\n"}, {'function_call': {'id': 'adk-c5300302-12f3-48bc-84a2-7f2910cbf4b5', 'args': {'query': 'coastal city colorful houses hills'}, 'name': 'search_places'}}, {'function_call': {'id': 'adk-2413b148-994c-4fc0-af3d-ec72373363f8', 'args': {'query': 'custard tarts'}, 'name': 'search_wikidata'}}, {'function_call': {'id': 'adk-d98d823f-c1a3-429e-b4bf-8e13b2ac90ac', 'args': {'query': 'city trams narrow streets'}, 'name': 'search_places'}}, {'function_call': {'id': 'adk-55e75ba9-daac-41c9-9ab0-acb791e03dfa', 'args': {'query': 'city street art'}, 'name': 'search_places'}}, {'function_call': {'id': 'adk-5a0deec9-22fe-43f7-bc6c-2345bd5edaf5', 'args': {'query': 'red bridge'}, 'name':

In [ ]:
# =============================================================
# CELL 10: Test — Greeting
# =============================================================

print("Test 4: Greeting")
print("=" * 60)
print()

for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Hello! How does this game work?",
):
    print(event)

Test 4: Greeting

{'model_version': 'gemini-2.0-flash', 'content': {'parts': [{'text': "Welcome to GeoGuesser Bot! 🌍\n\nI'll try to guess any place in the world based on your description!\n\nHOW TO PLAY:\n🗣️ Describe a place (landmarks, food, weather, culture, etc.)\n🔍 I'll research using Wikidata, OpenStreetMap, and country databases\n🌍 I'll make my best guess\n❌ Wrong? Give me more clues and I'll try again!\n✅ Right? Celebrate and play again!\n\nI can do up to 10 rounds of research per guess and I never give up!\nType 'quit' to end or 'new game' to start fresh.\n\nReady? Describe your place!\n"}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 140, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 140}], 'prompt_token_count': 1190, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 1190}], 'total_token_count': 1330, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': 5.478665116243064e-07, 'invocation_id': 'e-e2272cfc-d08a

In [ ]:
# =============================================================
# CELL 11: Validation Summary
# =============================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 5 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT 1: Create an agent using the ADK          ✅       ║
║  → GeoGuesser Bot with gemini-2.0-flash                          ║
║  → 3 knowledge tools: Wikidata, Nominatim, REST Countries        ║
║  → Internal search→guess→critique→refine workflow                ║
║  → Defined inside deploy function for clean serialization        ║
║                                                                  ║
║  REQUIREMENT 2: Deploy the agent to Agent Engine       ✅       ║
║  → AdkApp wrapper + agent_engines.create()                       ║
║  → Requirements: aiplatform + requests                           ║
║  → Previous deployments cleaned before creating                  ║
║                                                                  ║
║  REQUIREMENT 3: Test the agent                         ✅       ║
║  → Local test inside deploy function                             ║
║  → Remote tests: Easy (Paris), Medium (Uluru),                   ║
║    Hard (Lisbon), Greeting                                       ║
║                                                                  ║
║  DEPLOYMENT STEPS (from slides):                                 ║
║    Step 1: vertexai.init() — Cell 4                              ║
║    Step 2: Define agent — inside deploy function, Cell 6         ║
║    Step 3: Test locally — inside deploy function, Cell 6         ║
║    Step 4: agent_engines.create() — inside deploy function       ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

try:
    print(f"Deployed: {remote_agent.resource_name}")
    print("Status: OPERATIONAL")
except:
    print("WARNING: remote_agent not found — rerun Cell 6")


╔══════════════════════════════════════════════════════════════════╗
║           CHALLENGE 5 — REQUIREMENTS VALIDATION                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  REQUIREMENT 1: Create an agent using the ADK          ✅       ║
║  → GeoGuesser Bot with gemini-2.0-flash                          ║
║  → 3 knowledge tools: Wikidata, Nominatim, REST Countries        ║
║  → Internal search→guess→critique→refine workflow                ║
║  → Defined inside deploy function for clean serialization        ║
║                                                                  ║
║  REQUIREMENT 2: Deploy the agent to Agent Engine       ✅       ║
║  → AdkApp wrapper + agent_engines.create()                       ║
║  → Requirements: aiplatform + requests                           ║
║  → Previous deployments cleaned before creating                  ║
║                                    

In [ ]:
# =============================================================
# CELL 12: Cleanup (OPTIONAL)
# =============================================================

# Uncomment to delete and avoid charges:
# from vertexai import agent_engines
# agent_engines.delete(remote_agent.resource_name)
# print("Deleted.")

In [ ]:
# =============================================================
# CELL 13: Create a Cloud Run frontend for the GeoGuesser
# =============================================================

import subprocess
import os

AGENT_RESOURCE_NAME = remote_agent.resource_name
SERVICE_NAME = "geoguesser-frontend"

# Write a minimal Flask app
os.makedirs("frontend", exist_ok=True)

# app.py
with open("frontend/app.py", "w") as f:
    f.write('''
import os
from flask import Flask, request, jsonify, render_template_string
import vertexai
from vertexai import agent_engines

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ["GOOGLE_CLOUD_LOCATION"]
AGENT_RESOURCE = os.environ["AGENT_RESOURCE_NAME"]

vertexai.init(project=PROJECT_ID, location=LOCATION)

app = Flask(__name__)

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>GeoGuesser Bot</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body { font-family: -apple-system, BlinkMacSystemFont, sans-serif;
               background: #1a1a2e; color: #eee; height: 100vh;
               display: flex; flex-direction: column; }
        .header { background: #16213e; padding: 20px; text-align: center; }
        .header h1 { font-size: 24px; }
        .header p { color: #aaa; font-size: 14px; margin-top: 5px; }
        .chat { flex: 1; overflow-y: auto; padding: 20px; }
        .message { margin-bottom: 15px; max-width: 80%; padding: 12px 16px;
                   border-radius: 12px; line-height: 1.5; white-space: pre-wrap; }
        .user { background: #0f3460; margin-left: auto; text-align: right; }
        .bot { background: #16213e; border: 1px solid #333; }
        .input-area { display: flex; padding: 15px; background: #16213e;
                     border-top: 1px solid #333; }
        .input-area input { flex: 1; padding: 12px; border-radius: 8px;
                           border: 1px solid #333; background: #1a1a2e;
                           color: #eee; font-size: 16px; outline: none; }
        .input-area button { margin-left: 10px; padding: 12px 24px;
                            border-radius: 8px; border: none;
                            background: #e94560; color: white; font-size: 16px;
                            cursor: pointer; }
        .input-area button:hover { background: #c81e45; }
        .input-area button:disabled { background: #555; cursor: wait; }
        .typing { color: #aaa; font-style: italic; }
    </style>
</head>
<body>
    <div class="header">
        <h1>🌍 GeoGuesser Bot</h1>
        <p>Describe any place and I will try to guess it!</p>
    </div>
    <div class="chat" id="chat">
        <div class="message bot">Welcome to GeoGuesser Bot! 🌍 Describe any place — landmarks,
climate, culture, food, landscape — and I will investigate using
Wikidata, OpenStreetMap, and country databases.

I research, guess, critique myself, and refine! You can give me more hints if I'm struggling.
Ready? Describe your place!</div>
    </div>
    <div class="input-area">
        <input type="text" id="input" placeholder="Describe a place..."
               onkeypress="if(event.key==='Enter')sendMessage()">
        <button id="btn" onclick="sendMessage()">Send</button>
    </div>
    <script>
        const chat = document.getElementById('chat');
        const input = document.getElementById('input');
        const btn = document.getElementById('btn');

        async function sendMessage() {
            const msg = input.value.trim();
            if (!msg) return;

            // Show user message
            const userDiv = document.createElement('div');
            userDiv.className = 'message user';
            userDiv.textContent = msg;
            chat.appendChild(userDiv);
            input.value = '';
            btn.disabled = true;

            // Show typing indicator
            const typingDiv = document.createElement('div');
            typingDiv.className = 'message bot typing';
            typingDiv.textContent = '🔍 Researching...';
            chat.appendChild(typingDiv);
            chat.scrollTop = chat.scrollHeight;

            try {
                const resp = await fetch('/chat', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({message: msg})
                });
                const data = await resp.json();
                typingDiv.className = 'message bot';
                typingDiv.textContent = data.response || 'No response received.';
            } catch (e) {
                typingDiv.className = 'message bot';
                typingDiv.textContent = 'Error: ' + e.message;
            }
            btn.disabled = false;
            chat.scrollTop = chat.scrollHeight;
            input.focus();
        }
        input.focus();
    </script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(HTML)

@app.route("/chat", methods=["POST"])
def chat():
    user_message = request.json.get("message", "")
    try:
        remote = agent_engines.get(AGENT_RESOURCE)
        response_text = ""
        for event in remote.stream_query(
            user_id="web-user",
            message=user_message,
        ):
            if isinstance(event, dict):
                content = event.get("content", {})
                parts = content.get("parts", [])
                for part in parts:
                    if "text" in part:
                        response_text = part["text"]
        return jsonify({"response": response_text or "No response generated."})
    except Exception as e:
        return jsonify({"response": f"Error: {str(e)}"})

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    app.run(host="0.0.0.0", port=port)
''')

# requirements.txt
with open("frontend/requirements.txt", "w") as f:
    f.write("""flask==3.1.0
gunicorn==23.0.0
google-cloud-aiplatform[agent_engines]
""")

# Dockerfile
with open("frontend/Dockerfile", "w") as f:
    f.write("""FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD exec gunicorn --bind :$PORT --workers 1 --threads 2 --timeout 120 app:app
""")

# .dockerignore
with open("frontend/.dockerignore", "w") as f:
    f.write("__pycache__\n*.pyc\n.git\n")

print("Frontend files created in ./frontend/")
print("  app.py — Flask app with chat UI")
print("  requirements.txt — Python dependencies")
print("  Dockerfile — Container definition")

Frontend files created in ./frontend/
  app.py — Flask app with chat UI
  requirements.txt — Python dependencies
  Dockerfile — Container definition


In [ ]:
# =============================================================
# CELL 14: Deploy the frontend to Cloud Run
# =============================================================

import subprocess

SERVICE_NAME = "geoguesser-frontend"
AGENT_RESOURCE_NAME = remote_agent.resource_name

print(f"Deploying frontend to Cloud Run...")
print(f"  Service: {SERVICE_NAME}")
print(f"  Agent: {AGENT_RESOURCE_NAME}")
print(f"  Region: {LOCATION}")
print()
print("This may take 3-5 minutes...")
print()

result = subprocess.run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--source", "frontend",
    f"--project={PROJECT_ID}",
    f"--region={LOCATION}",
    "--allow-unauthenticated",
    "--set-env-vars",
    f"GOOGLE_CLOUD_PROJECT={PROJECT_ID},GOOGLE_CLOUD_LOCATION={LOCATION},AGENT_RESOURCE_NAME={AGENT_RESOURCE_NAME}",
    "--memory=512Mi",
    "--timeout=120",
    "--quiet",
], capture_output=True, text=True)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

if result.returncode == 0:
    # Extract the URL from the output
    for line in result.stderr.split("\n"):
        if "https://" in line:
            url = line.strip().split()[-1]
            print()
            print("=" * 60)
            print("FRONTEND DEPLOYED!")
            print("=" * 60)
            print(f"URL: {url}")
            print()
            print("Share this URL with anyone to play GeoGuesser Bot!")
            break
else:
    print()
    print("Deployment failed. Check stderr above for details.")

Deploying frontend to Cloud Run...
  Service: geoguesser-frontend
  Agent: projects/922001826607/locations/us-central1/reasoningEngines/6822978724233740288
  Region: us-central1

This may take 3-5 minutes...

STDOUT: 
STDERR: Building using Dockerfile and deploying container to Cloud Run service [geoguesser-frontend] in project [qwiklabs-gcp-01-171e30612222] region [us-central1]
Building and deploying...
Validating Service......done
Uploading sources.......done
Building Container....................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [ ]:
# =============================================================
# CELL 15: Print the URL again (in case you need it later)
# =============================================================

result = subprocess.run([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    f"--project={PROJECT_ID}",
    f"--region={LOCATION}",
    "--format=value(status.url)",
], capture_output=True, text=True)

url = result.stdout.strip()
if url:
    print(f"🌍 GeoGuesser Bot is live at:")
    print(f"   {url}")
    print()
    print("Share this URL with anyone to play!")
else:
    print("Could not retrieve URL. Check Cloud Run console.")

🌍 GeoGuesser Bot is live at:
   https://geoguesser-frontend-c6im27qhma-uc.a.run.app

Share this URL with anyone to play!
